# One-Scene Pilot Generator
This notebook generates a single randomized Blender scene and saves it to the ProceduralScenes folder.
It does not run scanning/export yet.

In [ ]:
import os
import random
import math
from datetime import datetime

try:
    import bpy
except ImportError as exc:
    raise RuntimeError("This notebook must run inside Blender's Python environment (bpy required).") from exc

# --- Pilot configuration (scene generation only) ---
SEED = 460
FORCE_ANOMALY = True
OUTPUT_DIR = r"C:\Users\grsha\Desktop\DAEN 460\scenes\ProceduralScenes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Scene bounds (50m x 50m)
X_MIN, X_MAX = -25.0, 25.0
Y_MIN, Y_MAX = -25.0, 25.0
Z_BASE = 0.0

# Randomization ranges (reasonable defaults for pilot)
LARGE_ROCK_COUNT_RANGE = (1, 2)
CLUSTER_COUNT_RANGE = (4, 7)
CLUSTER_SIZE_RANGE = (5, 10)
CLUSTER_RADIUS = 5.0
SCATTERED_SMALL_COUNT_RANGE = (10, 20)
TURTLE_CHANCE = 0.3
TURTLE_HEIGHT_RANGE = (1.0, 3.0)
ROCK_SCALE_RANGE = (0.7, 1.0)
SMALL_SCALE_RANGE = (0.8, 1.2)

ANOMALY_TYPES = {
    "boat": "tekne",
    "cube": "Cube",
    "sphere": "Mball",
}
LARGE_ROCKS = ["Rock0", "Rock6", "Rock3"]
SMALL_OBJECTS = ["10010_Coral_v1_L3", "Mesh_0"]
OPTIONAL_OBJECTS = ["10042_Sea_Turtle_V2_iterations-2"]
PRESERVE = ["Plane", "Camera", "Light", "TextureField"]

random.seed(SEED)
print(f"Seed set to: {SEED}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def name_matches(base_name, obj_name):
    return obj_name == base_name or obj_name.startswith(base_name + ".")

def random_position_in_area():
    return (random.uniform(X_MIN, X_MAX), random.uniform(Y_MIN, Y_MAX))

def random_position_in_cluster(center, radius):
    angle = random.uniform(0.0, 2.0 * math.pi)
    dist = random.uniform(0.0, radius)
    return (center[0] + dist * math.cos(angle), center[1] + dist * math.sin(angle))

def get_lowest_world_z(obj):
    if obj.type != "MESH" or not obj.data or len(obj.data.vertices) == 0:
        return obj.location.z
    world_vertices = [obj.matrix_world @ v.co for v in obj.data.vertices]
    return min(v.z for v in world_vertices)

def duplicate_object(obj_name):
    source = bpy.data.objects.get(obj_name)
    if source is None:
        return None, None

    dup = source.copy()
    if source.data:
        dup.data = source.data.copy()
    bpy.context.collection.objects.link(dup)
    dup.hide_viewport = False
    dup.hide_render = False
    return dup, source.rotation_euler.copy()

def place_object(obj, x, y, base_rotation, obj_type="small", turtle_mode=False):
    obj.location.x = x
    obj.location.y = y
    obj.location.z = 0.0

    if obj_type == "large_rock":
        obj.rotation_euler = (
            base_rotation[0] + random.uniform(-0.1, 0.1),
            base_rotation[1] + random.uniform(-0.1, 0.1),
            random.uniform(0.0, 2.0 * math.pi),
        )
        s = random.uniform(*ROCK_SCALE_RANGE)
        obj.scale = (s, s, s)
    else:
        obj.rotation_euler = (base_rotation[0], base_rotation[1], random.uniform(0.0, 2.0 * math.pi))
        if obj_type == "small":
            s = random.uniform(*SMALL_SCALE_RANGE)
            obj.scale = (s, s, s)
        else:
            obj.scale = (1.0, 1.0, 1.0)

    bpy.context.view_layer.update()

    if turtle_mode:
        obj.location.z = Z_BASE + random.uniform(*TURTLE_HEIGHT_RANGE)
    elif obj.type == "MESH" and obj.data and len(obj.data.vertices) > 0:
        lowest_z = get_lowest_world_z(obj)
        offset = lowest_z - obj.location.z
        burial = random.uniform(0.0, 0.5)
        obj.location.z = Z_BASE - offset - burial
    else:
        obj.location.z = Z_BASE

    bpy.context.view_layer.update()

# Detect objects from current open scene.
anomaly_objects = {"boat": [], "cube": [], "sphere": []}
large_rocks = []
small_objects = []
optional_objects = []
preserved_objects = []

for obj in bpy.data.objects:
    name = obj.name

    if any(key in name for key in PRESERVE):
        preserved_objects.append(name)
        continue

    detected_anomaly = False
    for anomaly_type, keyword in ANOMALY_TYPES.items():
        if name_matches(keyword, name):
            anomaly_objects[anomaly_type].append(name)
            detected_anomaly = True
            break
    if detected_anomaly:
        continue

    if any(name_matches(key, name) for key in LARGE_ROCKS):
        large_rocks.append(name)
        continue

    if any(name_matches(key, name) for key in OPTIONAL_OBJECTS):
        optional_objects.append(name)
        continue

    if any(name_matches(key, name) for key in SMALL_OBJECTS):
        small_objects.append(name)

print("Object discovery complete:")
print(f"  Preserved: {len(preserved_objects)}")
print(f"  Large rocks: {len(large_rocks)}")
print(f"  Small objects: {len(small_objects)}")
print(f"  Optional objects: {len(optional_objects)}")
print(f"  Anomalies total: {sum(len(v) for v in anomaly_objects.values())}")

# Remove previous procedural pilot objects and stale scan artifacts.
removed = 0
for obj in list(bpy.data.objects):
    if obj.get("procedural_pilot") or any(prefix in obj.name for prefix in ["noise_values", "real_values", "PathMarker", "ScanPath"]):
        bpy.data.objects.remove(obj, do_unlink=True)
        removed += 1
print(f"Removed objects from prior runs: {removed}")

In [ ]:
# Hide source objects used for duplication; keep preserved scene helpers visible.
all_anomaly_names = anomaly_objects["boat"] + anomaly_objects["cube"] + anomaly_objects["sphere"]
for obj_name in large_rocks + small_objects + optional_objects + all_anomaly_names:
    obj = bpy.data.objects.get(obj_name)
    if obj:
        obj.hide_viewport = True
        obj.hide_render = True

for obj_name in preserved_objects:
    obj = bpy.data.objects.get(obj_name)
    if obj:
        obj.hide_viewport = False
        obj.hide_render = False

placement_counts = {
    "large_rocks": 0,
    "small_clustered": 0,
    "small_scattered": 0,
    "optional": 0,
    "anomaly": 0,
}
placed_anomalies = []

# Place 1-2 large rocks.
if large_rocks:
    n_large = random.randint(*LARGE_ROCK_COUNT_RANGE)
    for _ in range(n_large):
        x, y = random_position_in_area()
        src = random.choice(large_rocks)
        dup, rot = duplicate_object(src)
        if dup:
            dup["procedural_pilot"] = True
            place_object(dup, x, y, rot, obj_type="large_rock")
            placement_counts["large_rocks"] += 1

# Place clustered and scattered small objects.
if small_objects:
    n_clusters = random.randint(*CLUSTER_COUNT_RANGE)
    for _ in range(n_clusters):
        center = random_position_in_area()
        n_cluster = random.randint(*CLUSTER_SIZE_RANGE)
        for _ in range(n_cluster):
            x, y = random_position_in_cluster(center, CLUSTER_RADIUS)
            src = random.choice(small_objects)
            dup, rot = duplicate_object(src)
            if dup:
                dup["procedural_pilot"] = True
                place_object(dup, x, y, rot, obj_type="small")
                placement_counts["small_clustered"] += 1

    n_scattered = random.randint(*SCATTERED_SMALL_COUNT_RANGE)
    for _ in range(n_scattered):
        x, y = random_position_in_area()
        src = random.choice(small_objects)
        dup, rot = duplicate_object(src)
        if dup:
            dup["procedural_pilot"] = True
            place_object(dup, x, y, rot, obj_type="small")
            placement_counts["small_scattered"] += 1

# Optional turtle-like objects.
if optional_objects and random.random() < TURTLE_CHANCE:
    x, y = random_position_in_area()
    src = random.choice(optional_objects)
    dup, rot = duplicate_object(src)
    if dup:
        dup["procedural_pilot"] = True
        place_object(dup, x, y, rot, obj_type="turtle", turtle_mode=True)
        placement_counts["optional"] += 1

# Force anomaly placement for pilot if available.
scene_has_anomaly = False
if FORCE_ANOMALY:
    available_types = [k for k, v in anomaly_objects.items() if v]
    if available_types:
        chosen_type = random.choice(available_types)
        n_anomaly = random.randint(1, 2)
        for _ in range(n_anomaly):
            src = random.choice(anomaly_objects[chosen_type])
            x, y = random_position_in_area()
            dup, rot = duplicate_object(src)
            if dup:
                dup["procedural_pilot"] = True
                place_object(dup, x, y, rot, obj_type="anomaly")
                placed_anomalies.append(src)
                placement_counts["anomaly"] += 1
                scene_has_anomaly = True
    else:
        print("No anomaly source objects found; generated scene will be clean.")

label = "anomaly" if scene_has_anomaly else "no_anomaly"
base_name = f"scene_001_{label}.blend"
save_path = os.path.join(OUTPUT_DIR, base_name)
if os.path.exists(save_path):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    stem, ext = os.path.splitext(base_name)
    save_path = os.path.join(OUTPUT_DIR, f"{stem}_{stamp}{ext}")

# copy=True writes a new .blend file without changing the current working scene file.
bpy.ops.wm.save_as_mainfile(filepath=save_path, copy=True)

generation_report = {
    "seed": SEED,
    "scene_has_anomaly": scene_has_anomaly,
    "placed_anomalies": placed_anomalies,
    "placement_counts": placement_counts,
    "output_path": save_path,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}

In [ ]:
if "generation_report" not in globals():
    raise RuntimeError("Run the previous cell first to generate a scene.")

print("One-scene pilot generation complete")
print(f"Seed: {generation_report['seed']}")
print(f"Anomaly scene: {generation_report['scene_has_anomaly']}")
print(f"Placed anomalies: {generation_report['placed_anomalies']}")
print("Placement counts:")
for k, v in generation_report["placement_counts"].items():
    print(f"  {k}: {v}")
print(f"Saved file: {generation_report['output_path']}")
print(f"Run time: {generation_report['timestamp']}")